# Relatório Técnico - Projeto Final: Biblioteca de Redes Neurais MLP

**INE5664 - Aprendizado de Máquina**<br/>
**Departamento de Informática e Estatística (INE/CTC/UFSC)**<br/>
**Professor:** Prof. Dr. Eduardo Camilo Inacio

---

### Identificação da Equipe

| Nome do Integrante | Matrícula |
| :--- | :--- |
| Lukas Dávila Resgalla | 23200519 |
| Leila Nunes dos Anjos | 23204361 |
| Giovanna Gomes Tonelli | 23205461 |

In [ ]:
SEMENTE = 42

In [ ]:
from sklearn.model_selection import train_test_split


def dividir_dados(X, y, semente, estratificar=False):
    """Divide em 70% treino, 15% validação e 15% teste."""
    estrato = y if estratificar else None
    X_treino, X_resto, y_treino, y_resto = train_test_split(
        X, y, test_size=0.30, random_state=semente, stratify=estrato
    )

    estrato = y_resto if estratificar else None
    X_val, X_teste, y_val, y_teste = train_test_split(
        X_resto, y_resto, test_size=0.50, random_state=semente, stratify=estrato
    )

    return X_treino, X_val, X_teste, y_treino, y_val, y_teste

In [ ]:
from sklearn.preprocessing import StandardScaler


def padronizar(X_treino, X_val, X_teste):
    """Padroniza os dados usando média e desvio calculados só no treino."""
    escalador = StandardScaler()
    X_treino = escalador.fit_transform(X_treino)
    X_val = escalador.transform(X_val)
    X_teste = escalador.transform(X_teste)
    return X_treino, X_val, X_teste

In [ ]:
import matplotlib.pyplot as plt


def plotar_curvas(historico, titulo):
    """Plota o custo de treino e o de validação ao longo das épocas."""
    plt.plot(historico["custo_treino"], label="Treino")
    if historico["custo_validacao"]:
        plt.plot(historico["custo_validacao"], label="Validação")
    plt.xlabel("Época")
    plt.ylabel("Custo")
    plt.title(titulo)
    plt.legend()
    plt.show()

## Regressão

Nesta seção usamos o dataset **Diabetes** do scikit-learn: 442 pacientes, 10 características (idade, sexo, IMC, pressão e medidas de sangue) e, como alvo, um número que mede a progressão da doença um ano depois. É um problema de **regressão**, porque o alvo é um valor contínuo.

Passos: dividir os dados em 70% treino, 15% validação e 15% teste; padronizar `X` e `y` usando só o treino; treinar uma rede com a nossa biblioteca `mlp/`; e avaliar no teste, depois de converter a previsão de volta à escala original.

In [ ]:
from sklearn.datasets import load_diabetes

dados = load_diabetes()
X = dados.data
y = dados.target

print("X:", X.shape)
print("y:", y.shape)

In [ ]:
y = y.reshape(-1, 1)

X_treino, X_val, X_teste, y_treino, y_val, y_teste = dividir_dados(
    X, y, SEMENTE
)

print("Treino:", X_treino.shape, y_treino.shape)
print("Validação:", X_val.shape, y_val.shape)
print("Teste:", X_teste.shape, y_teste.shape)

In [ ]:
X_treino, X_val, X_teste = padronizar(X_treino, X_val, X_teste)

escalador_y = StandardScaler()
y_treino = escalador_y.fit_transform(y_treino)
y_val = escalador_y.transform(y_val)
y_teste_original = y_teste
y_teste = escalador_y.transform(y_teste)

print("X treino: média", X_treino.mean().round(3), "desvio", X_treino.std().round(3))
print("y treino: média", y_treino.mean().round(3), "desvio", y_treino.std().round(3))

In [ ]:
from mlp.rede import RedeNeural
from mlp.camadas import Densa
from mlp.ativacoes import ReLU
from mlp.inicializadores import He
from mlp.custos import MSE

rede = RedeNeural(semente_aleatoria=SEMENTE)
rede.adicionar(Densa(10, 16, inicializador=He(), gerador_aleatorio=rede.gerador_aleatorio))
rede.adicionar(ReLU())
rede.adicionar(Densa(16, 8, inicializador=He(), gerador_aleatorio=rede.gerador_aleatorio))
rede.adicionar(ReLU())
rede.adicionar(Densa(8, 1, gerador_aleatorio=rede.gerador_aleatorio))
rede.compilar(MSE())

print("Camadas:", len(rede.camadas))
print("Saída para 3 exemplos:", rede.predizer(X_treino[:3]).shape)

In [ ]:
historico = rede.treinar(
    X_treino,
    y_treino,
    epocas=200,
    taxa_aprendizagem=0.01,
    tamanho_lote=32,
    X_validacao=X_val,
    y_validacao=y_val,
    verbal=True,
)

plotar_curvas(historico, "Regressão (Diabetes) - curvas de custo")

In [ ]:
import numpy as np

previsao_padronizada = rede.predizer(X_teste)
previsao = escalador_y.inverse_transform(previsao_padronizada)

erro = previsao - y_teste_original
mae = np.mean(np.abs(erro))
mse = np.mean(erro ** 2)
rmse = np.sqrt(mse)

# Referência: um modelo que sempre prevê a média do y de treino
rmse_referencia = np.sqrt(np.mean((y_teste_original - escalador_y.mean_) ** 2))

print(f"MAE:  {mae:.2f}")
print(f"MSE:  {mse:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"RMSE da referência (prever a média): {rmse_referencia:.2f}")

### Análise dos resultados

**Rede usada:** entrada com 10 valores, duas camadas escondidas (16 e 8 neurônios, ativação ReLU, inicialização He) e uma saída linear com 1 neurônio. Custo MSE, taxa de aprendizagem 0,01, lotes de 32 exemplos e 200 épocas, com semente 42.

**Curvas de custo:** o custo de treino cai durante todas as épocas. O custo de validação cai até cerca de 0,51 (por volta da época 110) e depois fica oscilando, sem melhorar. Isso indica um começo leve de *overfitting*: a rede continua se ajustando ao treino, mas deixa de ganhar na validação. Os custos do gráfico estão na escala padronizada do `y`.

**Resultado no teste (escala original):** as métricas estão na saída da célula acima. Em média, a previsão erra o valor real por cerca de 50 pontos (MAE), e o RMSE é de cerca de 60. Como referência, um modelo que sempre prevê a média do treino tem RMSE de cerca de 75. A rede, portanto, aprende algo útil, mas com margem moderada.

**Limitações:** o dataset é pequeno (442 pacientes) e tem bastante ruído, então não se espera um erro muito baixo. A arquitetura e os hiperparâmetros foram escolhidos de forma simples, olhando só o treino e a validação. O teste foi usado uma única vez, no fim.

**Possíveis melhorias:** parar o treino mais cedo (perto do menor custo de validação), testar uma rede menor, ou comparar outras taxas de aprendizagem e números de épocas.

## Classificação Binária

## Classificação Multiclasse

## Desafios e Aprendizados